# Rastro Lab: cómo Rastro explica una predicción de machine learning

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/germanbraun/Rastro/blob/main/lab/notebooks/rastro-lab.ipynb)

**El caso:** predecir cuánto gas va a dar un pozo nuevo de shale gas de Vaca Muerta (Argentina) en sus meses 2 a 7,
con datos públicos de la Secretaría de Energía.

**La idea:** se entrena **el mismo modelo** dos veces, con los mismos pozos, y solo cambia de dónde salen sus
variables:

| | Las variables salen de | Cada variable es |
|---|---|---|
| **A · sin Rastro** | las tablas crudas de la fuente | una columna |
| **B · con Rastro** | Rastro, la capa semántica | un concepto de negocio con su fuente, su regla y su evidencia |

**En este cuaderno:**

| Sección | Qué muestra |
|---|---|
| 1 · Los datos y el modelo | qué se carga, el algoritmo (una regresión lineal) y que todo coincide con lo que Rastro publicó |
| 2 · ¿Acierta mejor con Rastro? | no: A y B predicen igual de bien |
| 3 · La explicación | **lo que cambia:** la misma predicción explicada sin Rastro y con Rastro |
| 4 · Qué predicciones revisar | Rastro sabe cuáles se apoyan en algo dudoso |
| 5 · Una corrección | un dato corregido una vez en Rastro cambia lo que entra al modelo |
| 6 · Experimentar | todos los algoritmos probados, con una configuración que se puede cambiar |
| 7 · ¿Y con árboles? | qué aporta Rastro a un modelo de árboles (XGBoost), con ejemplos |

Corre sin instalar nada: lee una copia de lo que lee el modelo (`lab/notebooks/datos/`) y usa el mismo código que
la app de Rastro (`lab/gas/modelo.py`). Para ejecutarlo: *Entorno de ejecución → Ejecutar todas*.

In [1]:
# Find the repository (or clone it, in Colab) and import the same model code the app uses
import os
import subprocess
import sys
from pathlib import Path

root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "lab/gas/modelo.py").exists()), None)
if root is None:
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/germanbraun/Rastro.git"], check=True)
    root = Path("Rastro").resolve()
os.chdir(root)
sys.path.insert(0, str(root / "lab" / "gas"))

import hashlib
import json

import pandas as pd

import modelo  # noqa: E402

pd.set_option("display.max_colwidth", 120)
print("Código del modelo:", modelo.__file__.replace(str(root) + os.sep, ""), "· versión", modelo.version())

Código del modelo: lab/gas/modelo.py · versión 1b8209a1


## 1 · Los datos y el modelo

**Los datos** son una copia de lo que lee el modelo en Rastro, con la huella (SHA-256) de cada archivo, y las
predicciones que Rastro publicó con este mismo código.

In [2]:
DATA = root / "lab" / "notebooks" / "datos"
manifest = json.loads((DATA / "manifest.json").read_text())
for name, spec in manifest["files"].items():
    assert hashlib.sha256((DATA / name).read_bytes()).hexdigest() == spec["sha256"], f"{name} changed"
assert manifest["model_version"] == modelo.version(), "modelo.py changed after this snapshot: save a new one"

gas = pd.read_csv(DATA / "gas.csv", dtype={"idpozo": str})
raw = pd.read_csv(DATA / "a.csv", dtype={"idpozo": str})          # A: the raw tables' columns
rastro = pd.read_csv(DATA / "b.csv.gz")                            # B: Rastro's concepts, with rule and evidence
socios = pd.read_csv(DATA / "socios.csv")
cambios = pd.read_csv(DATA / "cambios.csv")
publicadas = pd.read_csv(DATA / "predicciones.csv", dtype={"idpozo": str})

A, B = modelo.preparar(gas, raw, rastro)
ctx = modelo.contexto(socios, cambios)
r = modelo.modelos(A, B)
print(f"Corrida publicada: {manifest['run']} · copia guardada el {manifest['saved_utc'][:10]}")
print(f"Pozos: {len(B)} · {(B.inicio < modelo.PRUEBA).sum()} para entrenar (arrancaron antes de 2025) · "
      f"{(B.inicio >= modelo.PRUEBA).sum()} para probar (arrancaron en 2025)")

Corrida publicada: 20261026-212547-1b8209a1 · copia guardada el 2026-10-28
Pozos: 559 · 510 para entrenar (arrancaron antes de 2025) · 49 para probar (arrancaron en 2025)


**El modelo** es una regresión lineal con un poco de regularización (*ridge*), elegida para que cada predicción
se pueda explicar **exacta**:

- **Variables:** la operadora y la concesión, como categorías (una columna 0/1 por cada una); y como números, la
  participación de la operadora, la fractura (rama horizontal, etapas, arena, agua y presión) y la profundidad. Un
  valor que falta se completa con el promedio del entrenamiento, así no suma ni resta.
- **Ajuste:** con las variables centradas en su promedio de entrenamiento $\bar{x}$ y el gas en el suyo $\bar{y}$,
  $$\beta = (X_c^\top X_c + \alpha I)^{-1} X_c^\top (y - \bar{y}), \qquad \alpha = 1.$$
- **Predicción:** $\hat{y} = \bar{y} + \sum_j (x_j - \bar{x}_j)\,\beta_j$. Cada término es el **aporte** de una
  variable, en millones de m³: la explicación no es una aproximación, es la cuenta misma.

El código, leído de `lab/gas/modelo.py`:

In [3]:
import inspect

print(inspect.getsource(modelo.ajustar))
print(inspect.getsource(modelo.entrenar))

def ajustar(df, categorias, numeros, prueba=PRUEBA, alpha=1.0):
    """El ajuste de la regresión ridge: la tabla de entrada (categorías en columnas 0/1, faltantes con el promedio
    del entrenamiento), sus promedios en el entrenamiento, el gas promedio y un coeficiente por columna."""
    entreno = (df.inicio < prueba).values
    x = pd.get_dummies(df[categorias].astype(str), prefix_sep="=", dtype=float)
    n = df[numeros].astype(float)
    n = n.fillna(n[entreno].mean())         # lo que falta, con el promedio del entrenamiento: no suma ni resta
    x = pd.concat([x, n], axis=1)
    media_x, media_y = x[entreno].mean(), df.gas[entreno].mean()
    xc = (x[entreno] - media_x).values
    coef = np.linalg.solve(xc.T @ xc + alpha * np.eye(xc.shape[1]), xc.T @ (df.gas[entreno].values - media_y))
    return x, n, entreno, media_x, media_y, coef

def entrenar(df, categorias, numeros, prueba=PRUEBA, alpha=1.0):
    """Regresión lineal con un poco de regularización (ridge). Devuelve la predic

**Tres comprobaciones**, para confiar en lo que sigue: el mismo ajuste con scikit-learn da las mismas
predicciones; la predicción de un pozo es exactamente el pozo promedio más sus aportes; y las predicciones son las
que Rastro publicó.

In [4]:
from sklearn.linear_model import Ridge

# 1. The same fit with scikit-learn's ridge regression
x, _, entreno, _, _, _ = modelo.ajustar(B, **modelo.VARIABLES["B"])
sk = Ridge(alpha=1.0).fit(x[entreno].values, B.gas[entreno].values)
dif_sklearn = abs(sk.predict(x.values) - r["B"]["pred"]).max()

# 2. One well's prediction is the average well plus its contributions
pozo = modelo.pozo_ejemplo(B)
i = B.index[B.idpozo == pozo][0]
suma = r["B"]["base"] + r["B"]["aportes"].loc[i].sum()

# 3. The predictions are the published run's (the database keeps them rounded)
juntas = publicadas.merge(pd.DataFrame({"idpozo": B.idpozo.values, "mia": r["B"]["pred"]}), on="idpozo")
dif_publicada = float((juntas.gas - juntas.mia).abs().max())

assert dif_sklearn < 1e-6 and abs(suma - r["B"]["pred"][i]) < 1e-9
assert len(juntas) == len(publicadas) and dif_publicada < 0.001
pd.DataFrame([
    {"Comprobación": "Igual a la ridge de scikit-learn", "Resultado": f"diferencia máxima {dif_sklearn:.0e} millones de m³"},
    {"Comprobación": f"Pozo {pozo}: promedio + aportes = predicción",
     "Resultado": f"{r['B']['base']:.2f} + {r['B']['aportes'].loc[i].sum():.2f} = {suma:.2f}"},
    {"Comprobación": "Iguales a las predicciones publicadas",
     "Resultado": f"{len(juntas)} de {len(publicadas)}, diferencia máxima {dif_publicada:.4f}"},
])

,Comprobación,Resultado
0,Igual a la ridge de scikit-learn,diferencia máxima 9e-13 millones de m³
1,Pozo 165404: promedio + aportes = predicción,51.64 + 2.51 = 54.15
2,Iguales a las predicciones publicadas,"559 de 559, diferencia máxima 0.0005"


> **En resumen:** un modelo simple, el mismo que usa la app, con números que se pueden verificar.

## 2 · ¿Acierta mejor con Rastro?

El error medio en los pozos de 2025, que el modelo no vio, en millones de m³. Como referencia, dos formas de
predecir sin modelo.

In [5]:
ref = modelo.referencias(A)
pd.DataFrame([
    {"Cómo se predice": "Sin modelo: la mediana de todos los pozos", "Error medio": ref["todos"]},
    {"Cómo se predice": "Sin modelo: la mediana de su concesión", "Error medio": ref["concesion"]},
    {"Cómo se predice": "A · el modelo, con las tablas crudas", "Error medio": r["A"]["error"], "R²": r["A"]["r2"]},
    {"Cómo se predice": "B · el modelo, con Rastro", "Error medio": r["B"]["error"], "R²": r["B"]["r2"]},
]).round(2)

,Cómo se predice,Error medio,R²
0,Sin modelo: la mediana de todos los pozos,22.35,NaN
1,Sin modelo: la mediana de su concesión,17.80,NaN
2,"A · el modelo, con las tablas crudas",16.65,0.46
3,"B · el modelo, con Rastro",16.67,0.46


> **En resumen:** no. A y B se equivocan casi lo mismo: Rastro no promete un modelo más preciso. Promete saber en
> qué se apoya cada predicción, que es lo que muestran las secciones 3 a 5.

## 3 · La explicación: una predicción, sin Rastro y con Rastro

El pozo de ejemplo es el que abre la pestaña Rastro Lab de la app. Primero, las dos explicaciones lado a lado.

In [6]:
sin = modelo.explicacion_a(r, A, pozo)
con = modelo.explicacion_b(r, B, ctx, pozo)
a_mirar = [h for h in con["hechos"] if h["marca"]]
pd.DataFrame({
    "Sin Rastro (A)": [f"{sin['pred']:.1f} millones de m³", f"{len(sin['filas'])} columnas, cada una con su aporte",
                       "no", "no"],
    "Con Rastro (B)": [f"{con['pred']:.1f} millones de m³", f"{len(con['factores'])} factores de negocio",
                       f"sí: {len(con['hechos'])} hechos, cada uno con su regla, evidencia y fuente",
                       f"sí: {len(a_mirar)} ({', '.join(h['hecho'].lower() for h in a_mirar)})"],
}, index=["Predicción", "Cómo se explica", "De dónde sale cada dato", "Qué mirar dos veces"])

,Sin Rastro (A),Con Rastro (B)
Predicción,54.1 millones de m³,54.1 millones de m³
Cómo se explica,"9 columnas, cada una con su aporte",3 factores de negocio
De dónde sale cada dato,no,"sí: 4 hechos, cada uno con su regla, evidencia y fuente"
Qué mirar dos veces,no,"sí: 2 (operadora, fractura)"


### Sin Rastro: columnas

Lo que da cualquier herramienta estándar: cuánto aporta cada columna. No dice de dónde salió cada valor, si alguien
lo corrigió, si la empresa es la misma en las dos tablas ni si el dato llegó tarde.

In [7]:
pd.DataFrame(sin["filas"]).sort_values("aporte", key=abs, ascending=False).round(2).rename(
    columns={"columna": "Columna de la tabla", "valor": "Valor", "aporte": "Aporte (millones de m³)"})

,Columna de la tabla,Valor,Aporte (millones de m³)
4,cantidad_fracturas,53,9.25
1,area,LCA,-7.12
3,longitud_rama_horizontal_m,3186,-4.55
6,agua_inyectada_m3,106919,2.35
2,porcentaje,50,1.96
0,empresa,PLUSPETROL S.A.,1.63
8,profundidad_vertical_mt,2895.06,-0.72
5,arena_tn,12471.8,-0.40
7,presion_maxima_psi,11472,0.07


### Con Rastro: factores de negocio, y los hechos que los respaldan

La misma cuenta, leída en conceptos: la concesión con su operadora y sus socios, cómo se fracturó el pozo y su
profundidad.

In [8]:
print(con["resumen"])
pd.DataFrame(con["factores"])[["factor", "aporte", "por_que"]].round(2).rename(
    columns={"factor": "Factor", "aporte": "Aporte (millones de m³)", "por_que": "Por qué"})

Se esperan 54,1 millones de m³: un pozo promedio da 51,6; estar en La Calera con Pluspetrol resta 3,6, cómo se fracturó suma 6,8 y su profundidad resta 0,7.


,Factor,Aporte (millones de m³),Por qué
0,"La Calera, con Pluspetrol",-3.57,"Los 73 pozos de La Calera que vio el modelo (todos de Pluspetrol) dieron una mediana de 45,5 millones de m³, menos q..."
1,Cómo se fracturó,6.80,"Pesan sobre todo: etapas de fractura 53 (típico: 39), +9,3; largo de la rama horizontal 3.186 m (típico: 2.278), −4,..."
2,Su profundidad,-0.72,"Está a 2.895 m de profundidad vertical (típico: 2.788), de su trayectoria; en el entrenamiento, los pozos más somero..."


Cada factor se apoya en **hechos de Rastro**. La columna «A mirar» dice cuáles no salen tal cual de la fuente: una
unión revisada por un asistente de IA que espera la confirmación de una persona, un dato corregido con un documento,
o una fractura cargada en el portal **después** de que el pozo arrancó (ese día, el modelo no la habría tenido).

In [9]:
pd.DataFrame(con["hechos"])[["hecho", "valor", "marca", "fuentes"]].rename(
    columns={"hecho": "Hecho", "valor": "Valor", "marca": "A mirar", "fuentes": "Fuente"})

,Hecho,Valor,A mirar,Fuente
0,Operadora,Pluspetrol,"unión revisada por IA, a confirmar por una persona",Capítulo IV de la Secretaría de Energía (declaraciones juradas)
1,Socios,"Pluspetrol 50 %, YPF S.A. 50 %",,Mapa de concesiones de la Secretaría de Energía (texto de socios)
2,Fractura,"3.186 m de rama, 53 etapas","unión revisada por IA, a confirmar por una persona · cargada 2 meses después de que arrancó el pozo","Datos de fractura de pozos (Adjunto IV), Secretaría de Energía"
3,Trayectoria,2.895 m de profundidad vertical,,"Trayectorias de pozo de Vaca Muerta, Secretaría de Energía"


Y de cada hecho, su regla y su evidencia, tal como Rastro las guarda:

In [10]:
for h in con["hechos"]:
    print(f"■ {h['hecho']}: {h['valor']}")
    print(f"  Regla: {h['regla']}")
    print(f"  Evidencia: {h['evidencia']}\n")

■ Operadora: Pluspetrol
  Regla: rige quien declara el pozo en su primer mes
  Evidencia: Primer mes con producción: 02/2025. Lo declara PLUSPETROL S.A como pozo gasífero, en LCA.

■ Socios: Pluspetrol 50 %, YPF S.A. 50 %
  Regla: texto de socios del mapa
  Evidencia: PLUSPETROL S.A.: 50 %

■ Fractura: 3.186 m de rama, 53 etapas
  Regla: las fracturas informadas hasta el mes en que arrancó el pozo, sumadas
  Evidencia: Fractura informada por PLUSPETROL S.A desde 04/10/2024: rama horizontal de 3186 m, 53 etapas, 12472 t de arena, 106919 m³ de agua, presión máxima 11472 psi. Cargada en el portal el 24/04/2025.

■ Trayectoria: 2.895 m de profundidad vertical
  Regla: una por pozo: la de rama más larga entre las perforadas antes de que arrancara
  Evidencia: Trayectoria PLU.Nq.LCa-3291(h): profundidad vertical de 2895 m, rama horizontal de 3209 m, perforado hasta el 27/06/2024.



> **En resumen:** sin Rastro, la explicación es una lista de columnas. Con Rastro, es una cuenta en palabras de
> negocio donde cada dato dice de dónde sale, con qué regla y qué evidencia, y cuáles hay que mirar dos veces.

## 4 · Qué predicciones revisar

Como cada predicción dice qué datos usó, se puede preguntar por todas a la vez cuáles se apoyan en algo dudoso.

In [11]:
dudosas, probadas = modelo.dudosos(B)
tarde = modelo.al_arrancar(r, B)
pd.DataFrame([
    {"Qué": "Pozos de 2025 predichos", "Cuántos": probadas},
    {"Qué": "…que se apoyan en una unión o un dato que no sale tal cual de la fuente", "Cuántos": dudosas},
    {"Qué": "…con datos de fractura", "Cuántos": tarde["con_fractura"]},
    {"Qué": "…cuya fractura se cargó después de que arrancó el pozo", "Cuántos": tarde["tarde"]},
    {"Qué": "Cambio medio de su predicción si se predice el día que arrancó (millones de m³)",
     "Cuántos": round(tarde["cambio_medio"], 1)},
])

,Qué,Cuántos
0,Pozos de 2025 predichos,49.0
1,…que se apoyan en una unión o un dato que no sale tal cual de la fuente,24.0
2,…con datos de fractura,43.0
3,…cuya fractura se cargó después de que arrancó el pozo,36.0
4,Cambio medio de su predicción si se predice el día que arrancó (millones de m³),5.1


> **En resumen:** con Rastro, revisar el modelo empieza por una lista concreta de predicciones y sus motivos. Sin
> Rastro no hay lista: las columnas no guardan de dónde vino cada valor ni cuándo llegó.

## 5 · Una corrección, hecha una vez, llega al modelo

En la concesión Aguada Pichana Este, el mapa de concesiones dice que Total tiene el 27,27 %. Un documento (el
Acuerdo Marco de Neuquén) dice 55 %. En Rastro se corrigió **una vez**, con el documento: el modelo con Rastro ya lo
usa, y el modelo sin Rastro sigue leyendo el texto del mapa. ¿Qué pasaría si se deshace la corrección?

In [12]:
corregidos = B.reglaSocio.map(modelo.marca) == modelo.texto("marca_documento")
ape = B[corregidos].merge(A[["idpozo", "porcentaje"]], on="idpozo")
print(f"Pozos con la participación corregida: {corregidos.sum()} · con Rastro: {ape.pct.iloc[0]:g} % · "
      f"sin Rastro (el mapa): {ape.porcentaje.iloc[0]:g} %")
print("Evidencia de la corrección:", B[corregidos].textoSocio.iloc[0][:300], "…")

# The model with Rastro, without the correction: the same B with the map's percentage
B_mapa = B.copy()
B_mapa.loc[corregidos, "pct"] = B_mapa.loc[corregidos, "idpozo"].map(A.set_index("idpozo").porcentaje)
sin_correccion = modelo.entrenar(B_mapa, **modelo.VARIABLES["B"])
cambio = pd.Series(r["B"]["pred"] - sin_correccion["pred"], index=B.idpozo)
de_prueba = (B.inicio >= modelo.PRUEBA).values
print(f"Predicciones de 2025 que cambian por la corrección: {(cambio[de_prueba].abs() > 0.005).sum()} de "
      f"{de_prueba.sum()} · el cambio más grande: {cambio[de_prueba].abs().max():.2f} millones de m³")

Pozos con la participación corregida: 74 · con Rastro: 55 % · sin Rastro (el mapa): 27.27 %
Evidencia de la corrección: Corrección revisada: TOTAL AUSTRAL S.A.: 55.0 %. El mapa de la Secretaría de Energía reparte la concesión en partes iguales (27,27 % para Total, Wintershall e YPF; 18,18 % para Pan American); el Acuerdo Marco lista 55 %, 22,5 %, 16,9 % y 5,6 %. Detectado en la verificación humana. Documento: Acuerdo …
Predicciones de 2025 que cambian por la corrección: 37 de 49 · el cambio más grande: 1.82 millones de m³


> **En resumen:** la corrección cambia lo que entra al modelo y lo que aprende, y la explicación de cada pozo de esa
> concesión la marca («dato corregido con documento»). Sin Rastro, el mismo arreglo habría que hacerlo en cada
> consulta y en el código del modelo.

## 6 · Experimentar con otros algoritmos

Todos los algoritmos que se probaron en Rastro Lab, con las mismas variables y el mismo corte por fecha. **Cambiá
la configuración de la primera celda y volvé a correr las dos.**

| Nombre | Algoritmo | De dónde sale |
|---|---|---|
| `mediana_concesion` | Sin modelo: la mediana del gas de los pozos de su concesión | referencia |
| `ridge` | **Regresión lineal ridge, el modelo publicado** (sección 1) | `lab/gas/modelo.py` |
| `arbol` | Un árbol de decisión chico (2 niveles, 10 pozos por hoja) | `lab/gas/compare_trees.py` |
| `bosque` | Bosque aleatorio, 500 árboles, 4 pozos por hoja | `lab/gas/compare_trees.py` |
| `rf`, `xgb`, `lgbm` | Bosque aleatorio, XGBoost y LightGBM con la configuración de la tesis, sobre el logaritmo del gas | `lab/gas/alternative.py` |
| `rf_direct`, `xgb_direct`, `lgbm_direct` | Los mismos tres, sobre el gas directo | `lab/gas/alternative.py` |

El *pipeline* completo de la tesis (`lab/gas/thesis_pipeline.py`) no está acá: necesita sus propias descargas; sus
resultados están en `docs/experiments/gas-thesis/`.

In [13]:
# The experiment's settings: change them, then run this cell and the next one
VARIABLES = ["A", "B"]          # "A": the raw tables' columns; "B": Rastro's concepts
CORTE = "2025-01-01"            # train with the wells that started before; test with the ones that started after
OCULTAR_FRACTURA_TARDIA = False # True: each well's fracturing as the source had it the month it started
ALPHA = 1.0                     # ridge's regularization (the published model uses 1.0)
ALGORITMOS = ["mediana_concesion", "ridge", "arbol", "bosque", "rf", "xgb", "lgbm",
              "rf_direct", "xgb_direct", "lgbm_direct"]

In [14]:
import time

import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor

import alternative
import compare_trees

corte = pd.Timestamp(CORTE)


def entrada(path):
    """The wells, with the features of path (A or B), as the model would see them."""
    frame = (A if path == "A" else B).copy()
    if OCULTAR_FRACTURA_TARDIA:
        frame, _ = compare_trees.available_inputs(frame, B.fracCargada,
                                                  modelo.FRACTURA_A if path == "A" else modelo.FRACTURA_B)
    return frame


def predecir(nombre, frame, path):
    """One algorithm's predictions for every well, trained only with the wells before the cutoff."""
    variables = modelo.VARIABLES[path]
    train = (frame.inicio < corte).values
    if nombre == "mediana_concesion":
        medianas = frame[train].groupby("area").gas.median()
        return frame.area.map(medianas).fillna(frame.gas[train].median()).to_numpy()
    if nombre == "ridge":
        return modelo.entrenar(frame, prueba=corte, alpha=ALPHA, **variables)["pred"]
    if nombre in ("arbol", "bosque"):
        x = compare_trees.design(frame, variables["categorias"], variables["numeros"], train)
        estimator = (DecisionTreeRegressor(max_depth=2, min_samples_leaf=10, random_state=42) if nombre == "arbol"
                     else RandomForestRegressor(n_estimators=500, min_samples_leaf=4, random_state=42, n_jobs=-1))
        return estimator.fit(x[train], frame.gas[train]).predict(x)
    return alternative.fit(frame, path, nombre, corte)[2]


filas = []
for path in VARIABLES:
    frame = entrada(path)
    prueba = (frame.inicio >= corte).values
    for nombre in ALGORITMOS:
        inicio = time.time()
        try:
            pred = predecir(nombre, frame, path)
        except ImportError as falta:   # XGBoost and LightGBM come with Colab; elsewhere: pip install xgboost lightgbm
            filas.append({"Variables": path, "Algoritmo": nombre, "Nota": f"falta instalar: {falta.name}"})
            continue
        y, p = frame.gas[prueba].to_numpy(), pred[prueba]
        filas.append({"Variables": path, "Algoritmo": nombre, "Error medio": float(np.abs(p - y).mean()),
                      "R²": modelo.r2(p, y), "Segundos": round(time.time() - inicio, 1)})

resultados = pd.DataFrame(filas)
print(f"Entrenamiento: pozos que arrancaron antes de {CORTE} · prueba: {prueba.sum()} pozos"
      + (" · fractura como estaba al arrancar" if OCULTAR_FRACTURA_TARDIA else ""))
resultados.round(2).sort_values(["Variables", "Error medio"]).reset_index(drop=True)

Entrenamiento: pozos que arrancaron antes de 2025-01-01 · prueba: 49 pozos


,Variables,Algoritmo,Error medio,R²,Segundos
0,A,xgb_direct,16.21,0.38,0.1
1,A,rf_direct,16.45,0.43,0.5
2,A,ridge,16.65,0.46,0.0
3,A,lgbm_direct,16.87,0.40,0.1
4,A,bosque,16.92,0.40,0.6
5,A,mediana_concesion,17.80,0.42,0.0
6,A,rf,18.59,0.21,0.6
7,A,xgb,19.09,0.17,0.1
8,A,lgbm,20.12,0.11,0.1
9,A,arbol,21.17,0.12,0.0


**Cómo leer la tabla:** el error medio está en millones de m³ de gas de los meses 2 a 7, en pozos que el modelo no
vio; el R² dice cuánto de la variación entre pozos explica cada modelo. Son pocos pozos de prueba, y diferencias
de medio millón de m³ cambian con la semilla y con el año de corte (con tres años, en `docs/experiments/gas-trees/`).

> **En resumen:** que un ensamble se equivoque un poco menos en una configuración no lo hace el modelo a publicar.
> **La regresión lineal se publica porque su predicción se descompone exacta en aportes**, y cada aporte se puede
> respaldar con la evidencia de Rastro (sección 3). Y en un mismo algoritmo, A y B dan casi lo mismo: Rastro no
> cambia cuánto acierta un modelo, cambia lo que se puede decir de cada predicción.

## 7 · ¿Y con árboles? Lo que aporta Rastro a cualquier modelo

Con árboles, Rastro tampoco cambia cuánto acierta el modelo (sección 6). Lo que aporta está en las **entradas**, no
en el algoritmo, así que vale igual para XGBoost que para la regresión lineal. Cuatro ejemplos con XGBoost sobre el
gas directo (`xgb_direct`), con las variables de Rastro.

In [15]:
import xgboost

xgb, x_xgb, pred_xgb = alternative.fit(B, "B", "xgb_direct", modelo.PRUEBA)
de_prueba = (B.inicio >= modelo.PRUEBA).values
print(f"XGBoost: error medio en los pozos de 2025 {np.abs(pred_xgb[de_prueba] - B.gas[de_prueba]).mean():.2f} "
      f"millones de m³ (ridge: {r['B']['error']:.2f})")

XGBoost: error medio en los pozos de 2025 16.12 millones de m³ (ridge: 16.67)


### Ejemplo 1 · La misma explicación por factores, con TreeSHAP

Un ensamble de árboles no se descompone solo en aportes: hace falta un método aparte. El estándar para árboles es
**TreeSHAP** (XGBoost lo trae, sin instalar nada): reparte cada predicción en un aporte por variable que suma
exacto. **SHAP responde cuánto aportó cada variable; Rastro, de dónde salió su valor y si se puede confiar en él.**
Juntos, la explicación de XGBoost queda tan respaldada como la de ridge.

In [16]:
contribs = xgb.get_booster().predict(xgboost.DMatrix(x_xgb.to_numpy()), pred_contribs=True)
shap = pd.DataFrame(contribs[:, :-1], columns=x_xgb.columns, index=B.index)   # the last column is the base value
i = B.index[B.idpozo == pozo][0]
assert abs(contribs[i].sum() - pred_xgb[i]) < 1e-3                         # it adds up to the prediction

# The same three business factors as ridge's explanation, each with the facts of Rastro that back it
factores = {"La concesión, con su operadora y socios": (["grupo", "area", "pct"], ["operadora", "socios"]),
            "Cómo se fracturó": (modelo.FRACTURA_B, ["fractura"]),
            "Su profundidad": (modelo.PROFUNDIDAD_B, ["trayectoria"])}
hechos = {h["id"]: h for h in con["hechos"]}


def aporte_shap(columnas):
    return float(sum(shap.loc[i, c] for c in shap.columns if c.split("=")[0] in columnas))


def aporte_ridge(columnas):
    return float(r["B"]["aportes"].loc[i, [c for c in r["B"]["aportes"].columns if c in columnas]].sum())


print(f"Pozo {pozo}: XGBoost predice {pred_xgb[i]:.1f} y ridge {r['B']['pred'][i]:.1f} millones de m³")
pd.DataFrame([{"Factor": factor, "Ridge": aporte_ridge(cols), "XGBoost (TreeSHAP)": aporte_shap(cols),
               "Hechos de Rastro": " · ".join(hechos[h]["valor"] for h in ids if h in hechos),
               "A mirar": " · ".join(hechos[h]["marca"] for h in ids if h in hechos and hechos[h]["marca"])}
              for factor, (cols, ids) in factores.items()]).round(2)

Pozo 165404: XGBoost predice 60.7 y ridge 54.1 millones de m³


,Factor,Ridge,XGBoost (TreeSHAP),Hechos de Rastro,A mirar
0,"La concesión, con su operadora y socios",-3.57,-5.12,"Pluspetrol · Pluspetrol 50 %, YPF S.A. 50 %","unión revisada por IA, a confirmar por una persona"
1,Cómo se fracturó,6.80,10.94,"3.186 m de rama, 53 etapas","unión revisada por IA, a confirmar por una persona · cargada 2 meses después de que arrancó el pozo"
2,Su profundidad,-0.72,3.29,2.895 m de profundidad vertical,


Los dos modelos reparten la predicción de forma distinta (cada uno aprendió lo suyo), pero los **hechos** detrás de
cada factor, y lo que hay que mirar dos veces, son los mismos: vienen de Rastro, no del modelo.

### Ejemplo 2 · Las predicciones de XGBoost a revisar, con su motivo

La misma pregunta de la sección 4, ahora para las predicciones de XGBoost, con sus tres motivos: una unión revisada
por IA que espera a una persona, un dato corregido con un documento, o una fractura cargada después de que arrancó
el pozo. No depende del algoritmo: depende de lo que entró.

In [17]:
def motivos(f):
    m = [modelo.marca(f.reglaNombre), modelo.marca(f.reglaSocio)]
    tarde = modelo.meses_tarde(f)
    m.append(modelo.texto("tarde_uno") if tarde == 1 else modelo.texto("tarde", n=tarde) if tarde else "")
    return " · ".join(x for x in dict.fromkeys(m) if x)


revisar = B[de_prueba].assign(**{"Predicción XGBoost": pred_xgb[de_prueba].astype(float), "Real": B.gas[de_prueba]})
revisar["Motivo"] = revisar.apply(motivos, axis=1)
revisar = revisar[revisar.Motivo != ""]
print(f"{len(revisar)} de {de_prueba.sum()} predicciones de XGBoost tienen algo que revisar")
revisar[["idpozo", "Predicción XGBoost", "Real", "Motivo"]].rename(columns={"idpozo": "Pozo"}).round(1).head(10)

47 de 49 predicciones de XGBoost tienen algo que revisar


,Pozo,Predicción XGBoost,Real,Motivo
512,165404,60.7,53.5,"unión revisada por IA, a confirmar por una persona · cargada 2 meses después de que arrancó el pozo"
513,165443,58.6,42.1,"unión revisada por IA, a confirmar por una persona · cargada 2 meses después de que arrancó el pozo"
514,165471,54.1,48.1,"unión revisada por IA, a confirmar por una persona · cargada 2 meses después de que arrancó el pozo"
515,165538,59.6,60.0,"unión revisada por IA, a confirmar por una persona · cargada 2 meses después de que arrancó el pozo"
516,166258,51.7,50.9,cargada 10 meses después de que arrancó el pozo
517,166263,52.7,64.6,cargada 10 meses después de que arrancó el pozo
518,166264,61.7,64.9,cargada 10 meses después de que arrancó el pozo
519,166265,46.1,44.6,cargada 10 meses después de que arrancó el pozo
520,165631,43.8,45.5,"unión revisada por IA, a confirmar por una persona · cargada 1 mes después de que arrancó el pozo"
521,165692,53.8,32.6,"unión revisada por IA, a confirmar por una persona · cargada 1 mes después de que arrancó el pozo"


### Ejemplo 3 · Sin datos del futuro

Rastro guarda cuándo se cargó cada fractura en el portal (`energia:loadedOn`). Con eso, cualquier modelo se puede
entrenar y probar con lo que **se sabía** el mes en que arrancó cada pozo, que es lo que habría tenido quien predijo
ese día. La fuente tiene el dato, pero la consulta sobre las tablas crudas (A) no lo usa: sin Rastro, cada consulta
y cada modelo tendrían que saberlo y repetirlo.

In [18]:
B_al_arrancar, ocultas = compare_trees.available_inputs(B, B.fracCargada, modelo.FRACTURA_B)
_, _, pred_al_arrancar = alternative.fit(B_al_arrancar, "B", "xgb_direct", modelo.PRUEBA)
pd.DataFrame([
    {"XGBoost entrenado con": "la fractura como está hoy en la fuente",
     "Error medio 2025": np.abs(pred_xgb[de_prueba] - B.gas[de_prueba]).mean()},
    {"XGBoost entrenado con": f"la fractura como estaba al arrancar cada pozo ({ocultas} pozos sin ella)",
     "Error medio 2025": np.abs(pred_al_arrancar[de_prueba] - B.gas[de_prueba]).mean()},
]).round(2)

,XGBoost entrenado con,Error medio 2025
0,la fractura como está hoy en la fuente,16.12
1,la fractura como estaba al arrancar cada pozo (478 pozos sin ella),16.29


Si el error con la fractura «al arrancar» es mayor, la diferencia es cuánto del acierto de hoy se apoya en datos que
llegaron después: un modelo puesto a predecir el día que arranca un pozo no los tendría.

### Ejemplo 4 · La corrección llega también a XGBoost

La participación de Total en Aguada Pichana Este, corregida una vez en Rastro (sección 5), entra igual a XGBoost.

In [19]:
_, _, pred_mapa = alternative.fit(B_mapa, "B", "xgb_direct", modelo.PRUEBA)
cambio_xgb = np.abs(pred_xgb - pred_mapa)[de_prueba]
print(f"Predicciones de XGBoost para 2025 que cambian por la corrección: {(cambio_xgb > 0.005).sum()} de "
      f"{de_prueba.sum()} · el cambio más grande: {cambio_xgb.max():.2f} millones de m³")

Predicciones de XGBoost para 2025 que cambian por la corrección: 49 de 49 · el cambio más grande: 1.21 millones de m³


> **En resumen:** con árboles, Rastro aporta lo mismo que con un modelo lineal, porque trabaja sobre las entradas:
> cada valor con su fuente, su regla y su evidencia; la lista de predicciones a revisar; datos sin futuro; y una
> corrección hecha una vez para todos los modelos. Lo que pierde un ensamble es la cuenta exacta de ridge: necesita
> TreeSHAP para repartir la predicción, y Rastro pone la evidencia al lado de cada aporte.

---
**Para verlo en vivo:** la app de Rastro, pestaña **Rastro Lab** (y el momento 3 de su demo), con cada dato abierto
hasta su registro de origen. Datos públicos de la Secretaría de Energía de Argentina, leídos a través de Rastro; las
uniones revisadas por IA esperan todavía la confirmación de una persona.